# Polaris GAM — parity preview, worked example

**Preview.** `polaris_re.gam` fits a *verified subset* of `mgcv`'s formula language by REML. It is not `mgcv`-compatible in general: see `docs/GAM_USER_GUIDE.md` for the subset, the refusal list, the oracle version and the gates, and `docs/GAM_PARITY_REPORT.md` for the generated comparison against `mgcv`. This notebook is the guide's §2, and the same formula and CSVs are fitted by `mgcv` in CI (report §4).

In [1]:
from polaris_re.gam import GUIDE_FAMILY, GUIDE_FORMULA, gam, load_guide_example

train, new = load_guide_example()
print(train.shape, new.shape)
print(GUIDE_FORMULA)
train.head()

(800, 5) (60, 4)
deaths ~ offset(log_exposure) + sex + s(age, k = 8, bs = 'cr') + s(duration, k = 6, bs = 'cr') + ti(age, duration, k = c(6, 4), bs = 'cr')


## Fit

One call, no solver options; the search is deterministic Newton REML.

In [2]:
fit = gam(GUIDE_FORMULA, train, GUIDE_FAMILY)
print(fit.converged)

True


## Summary

No p-values: they are outside the verified subset.

In [3]:
print(fit.summary())

Family: quasipoisson  Link function: log
Formula: deaths ~ offset(log_exposure) + sex + s(age, k = 8, bs = 'cr') + s(duration, k = 6, bs = 'cr') + ti(age, duration, k = c(6, 4), bs = 'cr')

Parametric coefficients: 2 (intercept included)

Smooth terms:
  term                          coef       edf   log10(sp)
  s(age)                           7    1.0053   7.690
  s(duration)                      5    4.9334   1.860
  ti(age,duration)                15    9.0804   1.849, 1.596

n = 800   total edf = 17.0192
scale est. = 8.85325
REML score = 2029.274807
Deviance explained = 76.749%  (deviance 6843.2, null 29432.3)
Newton REML search: 6 iterations, 7 penalised fits, relative projected gradient 7.689e-07 vs epsilon_rel 1e-06 — converged

No p-values are reported: the smooth-term tests of summary.gam (Wood 2013) are not in the verified preview subset.


## Predict at new rows, with standard errors

`unconditional=True` adds the smoothing-parameter-uncertainty correction (`Vc`).

In [4]:
link, link_se = fit.predict(new, "link", se_fit=True)
rate, rate_se = fit.predict(new, "response", se_fit=True)
_, link_se_c = fit.predict(new, "link", se_fit=True, unconditional=True)
for i in range(3):
    print(f"row {i}: rate {rate[i]:.4f}  se {rate_se[i]:.4f}  link se {link_se[i]:.4f}  (unconditional {link_se_c[i]:.4f})")

row 0: rate 22.9355  se 1.8079  link se 0.0788  (unconditional 0.0850)
row 1: rate 70.4816  se 4.2890  link se 0.0609  (unconditional 0.0645)
row 2: rate 97.3168  se 4.7583  link se 0.0489  (unconditional 0.0492)


## What is refused

Anything outside the verified subset is refused by name — here a bare `s(x)`, whose `mgcv` default basis (`tp`) is not yet verified.

In [5]:
from polaris_re.core.exceptions import PolarisValidationError

try:
    gam("deaths ~ s(age)", train, "quasipoisson")
except PolarisValidationError as exc:
    print(exc)

gam(): s(age) without bs= is not supported — a bare s(x) is mgcv's thin-plate default (bs='tp'), which is not verified yet (docs/MGCV_FEATURE_COVERAGE.md §2.1, ladder rung L7); write bs="cr" (or bs="re" for a factor)
